Skill extraction: dictionary x descriptions -> silver_posting_skills (Day 12).
Reads:  jobmarket.silver.silver_job_postings, skills_dictionary
Writes: jobmarket.silver.silver_posting_skills
Pattern: broadcast join, explode aliases, split-path matching
(contains vs boundary-regex per needs_boundary flag).

In [0]:
from pyspark.sql import functions as F
import re

# ---- The trap: aliases containing regex metacharacters ----
# "c++" as a regex = "c, one or more times". "c#", ".net", "a/b testing"
# all carry special chars. Any alias entering rlike() must be escaped.
# Python's re.escape() does exactly this - but we need it as a Spark-side
# operation. Simplest correct approach: escape in Python when building
# the pattern column, since aliases come from a 125-row table we can
# afford to handle driver-side.

# Test the concept on the known traps FIRST:
trap_tests = [
    # (description, alias, needs_boundary, should_match)
    ("we use c++ daily",              "c++",  True,  True),
    ("cc licensing experience",       "c++",  True,  False),  # naive regex would match!
    ("c# and .net stack",             "c#",   True,  True),
    ("strong c experience",           "c#",   True,  False),
    ("a/b testing programs",          "a/b testing", False, True),
    ("r and python required",         "r",    True,  True),
    ("senior engineer role",          "r",    True,  False),  # 'r' inside words
    ("we use sql server",             "sql",  True,  True),   # boundary still fires: "sql" starts "sql server"
    ("mysql administration",          "sql",  True,  False),  # no boundary before 'sql' in 'mysql'
    ("golang microservices",          "golang", False, True),
    ("scala development",             "scala", True, True),
    ("scalable systems",              "scala", True, False),  # THE trap from Day 11
]

def build_pattern(alias):
    """Escaped regex; word boundaries only where they can exist."""
    p = re.escape(alias)
    if alias[0].isalnum():         # open boundary only if first char is word-char
        p = r"\b" + p
    if alias[-1].isalnum():        # close boundary only if last char is word-char
        p += r"\b"
    return p

print(f"{'description':<28} {'alias':<12} {'expect':<7} {'got':<5} ok?")
all_pass = True
for desc, alias, needs_b, should in trap_tests:
    if needs_b:
        got = re.search(build_pattern(alias), desc) is not None
    else:
        got = alias in desc
    ok = got == should
    all_pass &= ok
    print(f"{desc:<28} {alias:<12} {str(should):<7} {str(got):<5} {'✓' if ok else '✗ FAIL'}")

assert all_pass, "Trap cases failed - fix before touching real data"
print("\n✅ All trap cases pass")

In [0]:
postings = (spark.table("jobmarket.silver.silver_job_postings")
    .filter(F.col("description").isNotNull())
    .select("posting_id", "posted_date", "source",
            F.lower(F.col("description")).alias("desc_lower")))

dictionary = spark.table("jobmarket.silver.skills_dictionary")

# explode aliases -> one row per (skill, alias); build patterns driver-side
# (125 skills is trivially collectible - this is NOT a pattern for big tables)
alias_rows = (dictionary
    .select("skill", "skill_group", "needs_boundary",
            F.explode("aliases").alias("alias"))
    .collect())

alias_data = [
    (r.skill, r.skill_group, r.alias,
     build_pattern(r.alias) if r.needs_boundary else None)
    for r in alias_rows
]
alias_df = spark.createDataFrame(
    alias_data, ["skill", "skill_group", "alias", "pattern"])

print(f"Alias rows: {alias_df.count()}")

# broadcast cross join + split-path match
matches = (postings
    .crossJoin(F.broadcast(alias_df))
    .filter(
        F.when(F.col("pattern").isNotNull(),
               F.expr("desc_lower rlike pattern"))       # boundary path
         .otherwise(F.col("desc_lower").contains(F.col("alias")))  # contains path
    ))

skills_out = (matches
    .select("posting_id", "skill", "skill_group",
            F.lit("dictionary").alias("match_method"), "posted_date")
    .dropDuplicates(["posting_id", "skill"]))

(skills_out.write.format("delta").mode("overwrite")
    .saveAsTable("jobmarket.silver.silver_posting_skills"))

print("Written:", spark.table("jobmarket.silver.silver_posting_skills").count(), "match rows")

In [0]:
sk = spark.table("jobmarket.silver.silver_posting_skills")
sp = spark.table("jobmarket.silver.silver_job_postings")

# coverage: how many postings matched at least one skill?
total_postings = sp.filter(F.col("description").isNotNull()).count()
matched_postings = sk.select("posting_id").distinct().count()
print(f"Postings with >=1 skill: {matched_postings} / {total_postings} "
      f"({matched_postings/total_postings*100:.1f}%)")

# avg skills per posting, split by source (the truncation asymmetry, measured)
(sk.join(sp.select("posting_id", "source"), "posting_id")
   .groupBy("source")
   .agg(F.countDistinct("posting_id").alias("postings_matched"),
        (F.count("*") / F.countDistinct("posting_id")).alias("avg_skills_per_posting"))
   .display())

# top 10 skills - the pre-audit preview
sk.groupBy("skill", "skill_group").count().orderBy(F.desc("count")).limit(10).display()
sk.filter(F.col("skill").isin("r", "workday")).groupBy("skill").count().display()